In [1]:
# ============================================================
# FUNCTION 3 — WEEK 6 CLEAN REBUILD
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

# ------------------------------------------------------------
# 1. Load original Function 3 data
# ------------------------------------------------------------

X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy").reshape(-1)

# ------------------------------------------------------------
# 2. Add Weeks 1–5 exactly once
# ------------------------------------------------------------

weekly_X = np.array([
    [0.433072, 0.324771, 0.537731],   # Week 1
    [0.364352, 0.404312, 0.451822],   # Week 2
    [0.987871, 0.206852, 0.790275],   # Week 3
    [0.514352, 0.554312, 0.412747],   # Week 4
    [0.926365, 0.997045, 0.456305]    # Week 5
])

weekly_Y = np.array([
    -0.022514823783463457,
    -0.014995206499628672,
    -0.09932090801789271,
    -0.019738170400010385,
    -0.056031037621799076
])

X = np.vstack([
    X,
    weekly_X
])

Y = np.concatenate([
    Y,
    weekly_Y
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Week 6 X shape:", X.shape)
print("Week 6 Y shape:", Y.shape)

print("\nBest observed input:")
print(best_x)

print("Best observed output:")
print(best_y)

print("\nWeek 5 input:")
print(weekly_X[-1])

print("Week 5 output:")
print(weekly_Y[-1])

Week 6 X shape: (20, 3)
Week 6 Y shape: (20,)

Best observed input:
[0.364352 0.404312 0.451822]
Best observed output:
-0.014995206499628672

Week 5 input:
[0.926365 0.997045 0.456305]
Week 5 output:
-0.056031037621799076


In [2]:
# ============================================================
# 3. FIT ARD GAUSSIAN PROCESS
# ============================================================

kernel = (
    C(
        1.0,
        (1e-3, 1e3)
    )
    *
    Matern(
        length_scale=[
            0.25,
            0.25,
            0.07
        ],
        length_scale_bounds=(
            0.005,
            3.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(
            1e-10,
            1e-2
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=63
)

gp.fit(
    X,
    Y
)

print("Fitted kernel:")
print(gp.kernel_)

lengthscales = np.asarray(
    gp.kernel_.k1.k2.length_scale
)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("\nLengthscales:")
print(lengthscales)

print("\nNormalised input influence:")

for i, value in enumerate(
    importance,
    start=1
):
    print(
        f"x{i}: {value:.4f}"
    )

Fitted kernel:
1.49**2 * Matern(length_scale=[0.701, 3, 0.215], nu=2.5) + WhiteKernel(noise_level=0.01)

Lengthscales:
[0.70089391 3.         0.21535958]

Normalised input influence:
x1: 0.2228
x2: 0.0521
x3: 0.7251


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 3.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.01. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [3]:
# ============================================================
# 4. WEEK 6 CANDIDATE GENERATION
#
# Focus mainly around Week 2 and Week 4.
# Week 5 was too far away and performed poorly.
# ============================================================

rng = np.random.default_rng(63)

week2_point = np.array([
    0.364352,
    0.404312,
    0.451822
])

week4_point = np.array([
    0.514352,
    0.554312,
    0.412747
])

# ------------------------------------------------------------
# A. Very local around current best — Week 2
# ------------------------------------------------------------

very_local = rng.normal(
    loc=week2_point,
    scale=[
        0.015,
        0.015,
        0.006
    ],
    size=(15_000, 3)
)

# ------------------------------------------------------------
# B. Local around Week 2
# ------------------------------------------------------------

local_best = rng.normal(
    loc=week2_point,
    scale=[
        0.040,
        0.040,
        0.012
    ],
    size=(15_000, 3)
)

# ------------------------------------------------------------
# C. Local around Week 4
# ------------------------------------------------------------

local_week4 = rng.normal(
    loc=week4_point,
    scale=[
        0.040,
        0.040,
        0.012
    ],
    size=(10_000, 3)
)

# ------------------------------------------------------------
# D. Blend between Week 2 and Week 4
# ------------------------------------------------------------

weights = rng.uniform(
    0,
    1,
    size=(10_000, 1)
)

blended = (
    weights * week2_point
    +
    (1 - weights) * week4_point
)

blended += rng.normal(
    0,
    [
        0.015,
        0.015,
        0.006
    ],
    size=(10_000, 3)
)

# ------------------------------------------------------------
# E. Small wider local search
#
# Still avoid huge jumps like Week 5.
# ------------------------------------------------------------

wider = rng.normal(
    loc=week2_point,
    scale=[
        0.080,
        0.080,
        0.025
    ],
    size=(7_500, 3)
)

candidates = np.vstack([
    very_local,
    local_best,
    local_week4,
    blended,
    wider
])

candidates = np.clip(
    candidates,
    0,
    1
)

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 57500


In [4]:
# ============================================================
# 5. REMOVE NEAR-DUPLICATES
# ============================================================

tree = cKDTree(X)

minimum_distance, _ = tree.query(
    candidates,
    k=1
)

keep = minimum_distance > 0.0025

candidates = candidates[keep]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 57440


In [5]:
# ============================================================
# 6. PREDICT CANDIDATE PERFORMANCE
# ============================================================

mean, std = gp.predict(
    candidates,
    return_std=True
)

# Expected Improvement
xi = 0.0005

improvement = (
    mean
    - best_y
    - xi
)

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        + std * norm.pdf(z)
    )

ei[std < 1e-12] = 0


# ------------------------------------------------------------
# Small UCB component
# ------------------------------------------------------------

kappa = 0.35

ucb = (
    mean
    + kappa * std
)

In [6]:
# ============================================================
# 7. FILTER OUT WEAK PREDICTED REGIONS
# ============================================================

mean_filter = (
    mean >= best_y - 0.008
)

# If too few survive, loosen slightly
if np.sum(mean_filter) < 100:
    mean_filter = (
        mean >= best_y - 0.015
    )

filtered_candidates = candidates[
    mean_filter
]

filtered_mean = mean[
    mean_filter
]

filtered_std = std[
    mean_filter
]

filtered_ei = ei[
    mean_filter
]

filtered_ucb = ucb[
    mean_filter
]

print(
    "Candidates passing mean filter:",
    len(filtered_candidates)
)

Candidates passing mean filter: 54839


In [7]:
# ============================================================
# 8. ACQUISITION SCORE
# ============================================================

ei_norm = (
    filtered_ei
    - filtered_ei.min()
) / (
    np.ptp(filtered_ei)
    + 1e-12
)

ucb_norm = (
    filtered_ucb
    - filtered_ucb.min()
) / (
    np.ptp(filtered_ucb)
    + 1e-12
)

# Week 6 is deliberately exploitation-heavy
acquisition = (
    0.90 * ei_norm
    +
    0.10 * ucb_norm
)

chosen_index = np.argmax(
    acquisition
)

week6_query = filtered_candidates[
    chosen_index
]

In [8]:
# ============================================================
# 9. FUNCTION 3 — WEEK 6 PORTAL OUTPUT
# ============================================================

print("\nSuggested Week 6 query:")
print(week6_query)

print("\nPortal format:")
print(
    "-".join(
        f"{value:.6f}"
        for value in week6_query
    )
)

print(
    "\nPredicted mean:",
    filtered_mean[chosen_index]
)

print(
    "Predicted uncertainty:",
    filtered_std[chosen_index]
)

print(
    "Expected Improvement:",
    filtered_ei[chosen_index]
)

print(
    "Distance from Week 2 best:",
    np.linalg.norm(
        week6_query
        - week2_point
    )
)

print(
    "Distance from Week 4:",
    np.linalg.norm(
        week6_query
        - week4_point
    )
)

print(
    "Distance from Week 5:",
    np.linalg.norm(
        week6_query
        - np.array([
            0.926365,
            0.997045,
            0.456305
        ])
    )
)


Suggested Week 6 query:
[0.45581519 0.11192948 0.44299337]

Portal format:
0.455815-0.111929-0.442993

Predicted mean: -0.014336585028758009
Predicted uncertainty: 0.019888584811934628
Expected Improvement: 0.008013960461083268
Distance from Week 2 best: 0.30648163996956346
Distance from Week 4: 0.4472624439068321
Distance from Week 5: 1.0025087568560598
